# Preprocess Datasets v4 (Per-Engine Balance, Realistic Timestamps, No Constant Dummies)

## Defined Configs and Import Libraries

In [1]:
# Config
PM_CSV = "data/predictive_maintenance.csv"
EA_CSV = "data/equipment_anomaly_data.csv"
ME_CSV = "data/marine_engine_data.csv"

ENCODING = "onehot"      # "onehot" -> SMOTE, "label" -> SMOTENC
BALANCE = "auto"         # "none", "smote", "smotenc", "auto"
TARGET_MODE = "status"   # "status", "failure", "health"
TIMESTAMP_FREQ = "D"     # "D" or "M"
OUT_DIR = "./processed"
SEED = 42

import os
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE, SMOTENC

warnings.filterwarnings("ignore")
os.makedirs(OUT_DIR, exist_ok=True)
print("[INFO] Config Loaded")

[INFO] Config Loaded


## Columns Specs

In [2]:
# Column maps
RENAME_PM = {
    "Air temperature [K]": "air_temperature",
    "Process temperature [K]": "process_temperature",
    "Rotational speed [rpm]": "rpm",
    "Torque [Nm]": "torque",
    "Tool wear [min]": "tool_wear",
    "Type": "type",
    "Failure Type": "failure_type",
}

RENAME_EA = {"temperature":"air_temperature","pressure":"pressure","vibration":"vibration","humidity":"humidity"}

UNIFIED_NUM_CANDIDATES = [
    "air_temperature","process_temperature","rpm","torque","tool_wear",
    "pressure","vibration","humidity",
    "engine_temp","oil_pressure","fuel_consumption","vibration_level","engine_load",
    "coolant_temp","exhaust_temp","running_period","fuel_consumption_per_hour"
]

UNIFIED_CAT_CANDIDATES = ["type","failure_type","equipment","location","failure_mode","fuel_type","engine_type","manufacturer"]

def to_datetime_safe(s): return pd.to_datetime(s, errors="coerce")
def detect_class_imbalance(y):
    vc = pd.Series(y).value_counts(normalize=True)
    return (len(vc) > 1) and (vc.min() < 0.3)
def map_status(series):
    return series.astype(str).map({"Normal":0,"Requires Maintenance":1,"Critical":2}).fillna(0).astype(int)
def infer_health_label_row(row):
    if "Target" in row and pd.notna(row["Target"]) and int(row["Target"])==1: return 1
    if "Failure Type" in row and isinstance(row["Failure Type"], str) and row["Failure Type"].lower() not in ["nan","none","no failure"]: return 1
    if "faulty" in row and pd.notna(row["faulty"]) and int(row["faulty"])==1: return 1
    if "maintenance_status" in row and isinstance(row["maintenance_status"], str) and row["maintenance_status"] in ["Requires Maintenance","Critical"]: return 1
    return 0
print("[INFO] Specs Ready")

[INFO] Specs Ready


## Data Loading

In [3]:

# Load
def load_pm(path):
    df = pd.read_csv(path); df.rename(columns=RENAME_PM, inplace=True); df["source"]="predictive_maintenance"; return df
def load_ea(path):
    df = pd.read_csv(path); df.rename(columns=RENAME_EA, inplace=True); df["source"]="equipment_anomaly"; return df
def load_me(path):
    df = pd.read_csv(path); df["timestamp"]=to_datetime_safe(df["timestamp"]); df["source"]="marine_engine"; return df

try:
    df_pm = load_pm(PM_CSV)
    df_ea = load_ea(EA_CSV)
    df_me = load_me(ME_CSV)
    print("[RESULT] Loaded Shapes:", df_pm.shape, df_ea.shape, df_me.shape)
except Exception as e:
    print("[ERROR] Load Failed:", e); raise

display(df_pm.head()); display(df_ea.head()); display(df_me.head())

[RESULT] Loaded Shapes: (10000, 11) (7672, 8) (5200, 18)


,UDI,Product ID,type,air_temperature,process_temperature,rpm,torque,tool_wear,Target,failure_type,source
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure,predictive_maintenance
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure,predictive_maintenance
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure,predictive_maintenance
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure,predictive_maintenance
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure,predictive_maintenance


,air_temperature,pressure,vibration,humidity,equipment,location,faulty,source
0,58.180180,25.029278,0.606516,45.694907,Turbine,Atlanta,0.0,equipment_anomaly
1,75.740712,22.954018,2.338095,41.867407,Compressor,Chicago,0.0,equipment_anomaly
2,71.358594,27.276830,1.389198,58.954409,Turbine,San Francisco,0.0,equipment_anomaly
3,71.616985,32.242921,1.770690,40.565138,Pump,Atlanta,0.0,equipment_anomaly
4,66.506832,45.197471,0.345398,43.253795,Pump,New York,0.0,equipment_anomaly


,timestamp,engine_id,engine_temp,oil_pressure,fuel_consumption,vibration_level,rpm,engine_load,coolant_temp,exhaust_temp,running_period,fuel_consumption_per_hour,maintenance_status,failure_mode,engine_type,fuel_type,manufacturer,source
0,2023-01-01,ENG_001,79.816406,7.049409,1000.000000,4.366612,1770.214578,42.472407,78.323108,450.0,49.741791,100.0,Critical,Oil Leakage,4-stroke High-Speed,Diesel,MAN B&W,marine_engine
1,2023-01-08,ENG_001,98.982068,8.000000,6308.623817,3.732792,1677.238238,77.042858,100.000000,450.0,94.351515,100.0,Requires Maintenance,Oil Leakage,2-stroke Low-Speed,Diesel,Mitsubishi,marine_engine
2,2023-01-15,ENG_001,83.918153,8.000000,6444.402260,4.061372,1487.472085,63.919637,78.178337,450.0,120.095804,100.0,Normal,No Failure,2-stroke Medium-Speed,Diesel,Caterpillar,marine_engine
3,2023-01-22,ENG_001,81.887081,7.601603,4439.946613,3.999554,1548.624692,55.919509,82.896344,450.0,122.321555,100.0,Requires Maintenance,Mechanical Wear,2-stroke Medium-Speed,Diesel,MAN B&W,marine_engine
4,2023-01-29,ENG_001,78.550429,6.233033,3146.234038,4.520559,1441.151499,29.361118,80.791150,450.0,111.978460,100.0,Normal,No Failure,4-stroke High-Speed,Diesel,Wärtsilä,marine_engine


## Preprocessing

In [4]:
# Build combined resampled table (no encoding)
def synthesize_timestamps(n, low, high, seed=42):
    rng = np.random.default_rng(seed)
    start = pd.Timestamp(low); end = pd.Timestamp(high)
    delta = (end - start).total_seconds()
    if delta <= 0: return pd.date_range(start, periods=n, freq="D")
    offs = rng.uniform(0, delta, size=n); ts = [start + pd.to_timedelta(x, unit="s") for x in offs]
    return pd.to_datetime(ts)

# Ensure timestamp in PM and EA
ts_min, ts_max = df_me["timestamp"].min(), df_me["timestamp"].max()
if "timestamp" not in df_pm.columns: df_pm["timestamp"] = synthesize_timestamps(len(df_pm), ts_min, ts_max, SEED)
else: df_pm["timestamp"] = to_datetime_safe(df_pm["timestamp"])
if "timestamp" not in df_ea.columns: df_ea["timestamp"] = synthesize_timestamps(len(df_ea), ts_min, ts_max, SEED+7)
else: df_ea["timestamp"] = to_datetime_safe(df_ea["timestamp"])

# engine_id existence
if "engine_id" not in df_me.columns: df_me["engine_id"] = "engine_0"
df_pm["engine_id"] = df_pm.get("engine_id", pd.Series([f"pm_{i}" for i in range(len(df_pm))]))
df_ea["engine_id"] = df_ea.get("engine_id", pd.Series([f"ea_{i}" for i in range(len(df_ea))]))

# targets
for d in [df_pm, df_ea, df_me]:
    d["health_label"] = d.apply(infer_health_label_row, axis=1).astype(int)
df_me["maintenance_status_num"] = map_status(df_me["maintenance_status"].astype(str)) if "maintenance_status" in df_me.columns else 0

if TARGET_MODE=="status":
    df_pm["unified_target"] = df_pm["health_label"]; df_ea["unified_target"] = df_ea["health_label"]; df_me["unified_target"] = df_me["maintenance_status_num"]
elif TARGET_MODE=="failure":
    df_pm["unified_target"] = df_pm.get("failure_mode", df_pm.get("failure_type","None")).astype(str)
    df_ea["unified_target"] = "None"
    df_me["unified_target"] = df_me.get("failure_mode","None").astype(str)
    le_t = LabelEncoder()
    all_t = pd.concat([df_pm["unified_target"].astype(str), pd.Series(df_ea["unified_target"]).astype(str), df_me["unified_target"].astype(str)], axis=0)
    le_t.fit(all_t)
    df_pm["unified_target"] = le_t.transform(df_pm["unified_target"].astype(str))
    df_ea["unified_target"] = le_t.transform(pd.Series(df_ea["unified_target"]).astype(str))
    df_me["unified_target"] = le_t.transform(df_me["unified_target"].astype(str))
else:
    df_pm["unified_target"] = df_pm["health_label"]; df_ea["unified_target"] = df_ea["health_label"]; df_me["unified_target"] = df_me["health_label"]

def keep_cols(df):
    base = ["timestamp","engine_id","unified_target"]
    cats = [c for c in UNIFIED_CAT_CANDIDATES if c in df.columns]
    nums = [c for c in UNIFIED_NUM_CANDIDATES if c in df.columns]
    return df[base+cats+nums].copy(), cats, nums

pm_keep, pm_cats, pm_nums = keep_cols(df_pm)
ea_keep, ea_cats, ea_nums = keep_cols(df_ea)
me_keep, me_cats, me_nums = keep_cols(df_me)

all_cat_union = sorted(list(set(pm_cats + ea_cats + me_cats)))
all_num_union = sorted(list(set(pm_nums + ea_nums + me_nums)))

raw_all = pd.concat([pm_keep, ea_keep, me_keep], ignore_index=True).sort_values(["engine_id","timestamp"])
raw_all["timestamp"] = to_datetime_safe(raw_all["timestamp"])

frames = []
freq = TIMESTAMP_FREQ.upper()
for eng, g in raw_all.groupby("engine_id"):
    g = g.set_index("timestamp").sort_index()
    g_num = g[all_num_union].resample(freq).mean()
    g_cat = g[all_cat_union].resample(freq).last()
    g_tgt = g[["unified_target"]].resample(freq).last()
    g_res = pd.concat([g_num, g_cat, g_tgt], axis=1)
    g_res["engine_id"] = eng
    frames.append(g_res.reset_index())

combined_resampled = pd.concat(frames, ignore_index=True).sort_values(["engine_id","timestamp"]).reset_index(drop=True)
combined_resampled["unified_target"] = combined_resampled.groupby("engine_id", group_keys=False)["unified_target"].apply(lambda s: s.ffill().bfill())
print("[INFO] Resampled Combined Shape:", combined_resampled.shape)

[INFO] Resampled Combined Shape: (53772, 28)


## Balancing Datasets

In [5]:
# ---- Per-engine balancing with realistic timestamps ----
def encode_features(X, cat_cols_present, encoding):
    if encoding == "label":
        X_enc = X.copy()
        for c in cat_cols_present:
            le = LabelEncoder()
            X_enc[c] = le.fit_transform(X_enc[c].astype(str).fillna("NA"))
        return X_enc, {"cat_idx":[X_enc.columns.get_loc(c) for c in cat_cols_present]}
    else:
        # onehot, avoid dummy_na to prevent *_nan constants
        X_enc = pd.get_dummies(X, columns=cat_cols_present, dummy_na=False)
        # drop constant columns
        nunique = X_enc.nunique(dropna=False)
        keep_cols = nunique[nunique > 1].index.tolist()
        if len(keep_cols) != X_enc.shape[1]:
            X_enc = X_enc[keep_cols]
        return X_enc, {}

def balance_one_engine(g, encoding, balance, seed=42):
    # g has columns: timestamp, engine_id, unified_target, cats, nums
    X = g.drop(columns=["timestamp","engine_id","unified_target"]).copy()
    y = g["unified_target"].astype(int)

    # impute numeric means then 0 for leftovers
    X = X.fillna(X.mean(numeric_only=True)).fillna(0)

    cat_present = [c for c in all_cat_union if c in X.columns]
    X_enc, extra = encode_features(X, cat_present, encoding)

    # if no imbalance or balance disabled
    if not detect_class_imbalance(y) or balance == "none":
        return X_enc, y, g["timestamp"].values

    # choose sampler
    if encoding == "label":
        cat_idx = extra.get("cat_idx", [])
        if len(cat_idx) and balance in ["auto","smotenc"]:
            sm = SMOTENC(categorical_features=cat_idx, random_state=seed)
        else:
            sm = SMOTE(random_state=seed)
        Xb, yb = sm.fit_resample(X_enc, y)
    else:
        sm = SMOTE(random_state=seed) if balance in ["auto","smote"] else None
        if sm is not None:
            Xb, yb = sm.fit_resample(X_enc, y)
        else:
            Xb, yb = X_enc, y

    # reconstruct timestamps for this engine
    n_bal = len(yb)
    tmin, tmax = g["timestamp"].min(), g["timestamp"].max()
    if n_bal <= len(g):
        # sample without replacement to keep realistic points
        ts_bal = np.sort(np.random.default_rng(seed).choice(g["timestamp"].values, size=n_bal, replace=False))
    else:
        # generate evenly spaced grid within engine range
        ts_bal = pd.to_datetime(np.linspace(tmin.value, tmax.value, n_bal))

    return pd.DataFrame(Xb, columns=X_enc.columns), pd.Series(yb), ts_bal

# run per engine
eng_results = []
for eng, g in combined_resampled.groupby("engine_id"):
    Xb, yb, tsb = balance_one_engine(g, ENCODING, BALANCE, SEED)
    dfb = pd.DataFrame(Xb, columns=Xb.columns).copy()
    dfb["timestamp"] = tsb
    dfb["engine_id"] = eng
    dfb["unified_target"] = yb.values
    eng_results.append(dfb)

balanced_all = pd.concat(eng_results, ignore_index=True)
print("[INFO] Per-engine Balanced Shape:", balanced_all.shape)

[INFO] Per-engine Balanced Shape: (57796, 33)


## Saved Datasets

In [6]:
# Save artifacts
comb_dir = Path(OUT_DIR) / "combined"; comb_dir.mkdir(parents=True, exist_ok=True)
combined_unbal_path = comb_dir / f"combined_{TIMESTAMP_FREQ.upper()}_processed_v4_per_engine.csv"
combined_bal_path = comb_dir / f"combined_{TIMESTAMP_FREQ.upper()}_processed_balanced_v4_per_engine.csv"

combined_resampled.to_csv(combined_unbal_path, index=False)
balanced_all.sort_values(["engine_id","timestamp"]).to_csv(combined_bal_path, index=False)

print("[RESULT] Saved Unbalanced to", combined_unbal_path)
print("[RESULT] Saved Balanced (per-engine, realistic timestamps) to", combined_bal_path)

[RESULT] Saved Unbalanced to processed\combined\combined_D_processed_v4_per_engine.csv
[RESULT] Saved Balanced (per-engine, realistic timestamps) to processed\combined\combined_D_processed_balanced_v4_per_engine.csv


In [7]:
new_df = pd.read_csv("processed/combined/combined_D_processed_balanced_v4_per_engine.csv")
new_df.head()

,coolant_temp,engine_load,engine_temp,exhaust_temp,fuel_consumption,fuel_consumption_per_hour,oil_pressure,rpm,running_period,vibration_level,...,manufacturer_0,manufacturer_Caterpillar,manufacturer_MAN B&W,manufacturer_Mitsubishi,manufacturer_Rolls-Royce,manufacturer_Wärtsilä,manufacturer_Yanmar,timestamp,engine_id,unified_target
0,78.323108,42.472407,79.816406,450.000000,1000.000000,100.00000,7.049409,1770.214578,49.741791,4.366612,...,False,False,True,False,False,False,False,2023-01-01 00:00:00.000000000,ENG_001,2
1,84.198137,47.985870,84.468660,449.855566,3953.949328,125.85818,7.154358,1485.517805,89.088485,3.696642,...,True,False,False,False,False,False,False,2023-01-02 00:00:00.000000000,ENG_001,2
2,84.198137,47.985870,84.468660,449.855566,3953.949328,125.85818,7.154358,1485.517805,89.088485,3.696642,...,True,False,False,False,False,False,False,2023-01-03 00:00:00.000000000,ENG_001,2
3,84.198137,47.985870,84.468660,449.855566,3953.949328,125.85818,7.154358,1485.517805,89.088485,3.696642,...,True,False,False,False,False,False,False,2023-01-04 00:00:00.000000000,ENG_001,2
4,84.198137,47.985870,84.468660,449.855566,3953.949328,125.85818,7.154358,1485.517805,89.088485,3.696642,...,True,False,False,False,False,False,False,2023-01-05 00:00:00.000000000,ENG_001,2


## Assessing New Blaanced Dataset

In [8]:
def assess_dataframe(df, target=None, high_card_threshold=0.5):
    print("Data Assessment Report")

    n_rows, n_cols = df.shape
    print(f"\nShape of Dataset: {n_rows} Rows x {n_cols} Columns")

    print("\nColumn Information:")
    info_table = pd.DataFrame({
        "DataType": df.dtypes,
        "Non-Null Count": df.notnull().sum(),
        "Null Count": df.isnull().sum(),
        "Null %": (df.isnull().sum() / n_rows * 100).round(2),
        "Unique Values": df.nunique(dropna=True),
        "Unique %": (df.nunique(dropna=True) / n_rows * 100).round(2),
    })
    print(info_table)

    dup_count = df.duplicated().sum()
    dup_percent = round((dup_count / n_rows) * 100, 2) if n_rows else 0.0
    print(f"\nDuplicate Rows: {dup_count} ({dup_percent}%)")

    const_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    print("\nConstant Columns:")
    print(const_cols if const_cols else "None")

    cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    high_card_cols = []
    for c in cat_cols:
        ratio = (df[c].nunique(dropna=True) / n_rows) if n_rows else 0.0
        if ratio >= high_card_threshold:
            high_card_cols.append((c, round(ratio*100, 2)))
    print("\nHigh Cardinality Categorical Columns (>= {:.0f}% Unique):".format(high_card_threshold*100))
    if high_card_cols:
        for c, pct in high_card_cols:
            print(f"- {c}: {pct}% Unique")
    else:
        print("None")

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        print("\nNumeric Columns Summary:")
        modes = {}
        if len(num_cols) > 0:
            mode_df = df[num_cols].mode(dropna=True)
            for c in num_cols:
                modes[c] = mode_df[c].iloc[0] if not mode_df.empty and c in mode_df else np.nan
        summary = pd.DataFrame({
            "Mean": df[num_cols].mean(numeric_only=True),
            "Median": df[num_cols].median(numeric_only=True),
            "Mode": pd.Series(modes),
            "Min": df[num_cols].min(numeric_only=True),
            "Q1": df[num_cols].quantile(0.25, numeric_only=True),
            "Q3": df[num_cols].quantile(0.75, numeric_only=True),
            "Max": df[num_cols].max(numeric_only=True),
            "Standard Deviation": df[num_cols].std(numeric_only=True),
            "Skewness": df[num_cols].skew(numeric_only=True),
            "Kurtosis": df[num_cols].kurt(numeric_only=True)
        }).round(4)
        print(summary)

        print("\nOutlier Counts (IQR method):")
        outlier_rows = []
        for c in num_cols:
            series = df[c].dropna()
            if series.empty:
                outlier_rows.append((c, 0, 0.0))
                continue
            q1 = series.quantile(0.25)
            q3 = series.quantile(0.75)
            iqr = q3 - q1
            if iqr == 0:
                outlier_rows.append((c, 0, 0.0))
                continue
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            count = ((series < lower) | (series > upper)).sum()
            pct = round(count / len(series) * 100, 2)
            outlier_rows.append((c, int(count), pct))
        if outlier_rows:
            outlier_table = pd.DataFrame(outlier_rows, columns=["Column", "Outliers", "Outliers %"])
            print(outlier_table)

        print("\nPearson Correlation (Numeric):")
        pearson_corr = df[num_cols].corr(method="pearson")
        print(pearson_corr.round(3))

        print("\nSpearman Correlation (Numeric):")
        spearman_corr = df[num_cols].corr(method="spearman")
        print(spearman_corr.round(3))
    else:
        print("\nNo Numeric columns found.")

    print("\nColumns > 50% Missing Values:")
    high_missing = info_table[info_table["Null %"] > 50].sort_values("Null %", ascending=False)
    if not high_missing.empty:
        print(high_missing[["Null Count", "Null %"]])
    else:
        print("None")

    print("\nMemory Usage:")
    total_mem = df.memory_usage(deep=True).sum()
    print(f"Total: {total_mem} bytes ({round(total_mem/1024,2)} KB, {round(total_mem/(1024*1024),2)} MB)")
    memory_per_col = df.memory_usage(deep=True)
    memory_table = pd.DataFrame({"Bytes": memory_per_col}).sort_values("Bytes", ascending=False)
    print("\nPer Column Memory:")
    print(memory_table)

    if target is not None and target in df.columns:
        print(f"\nClass Balance for '{target}':")
        vc = df[target].value_counts(dropna=False)
        vp = df[target].value_counts(normalize=True, dropna=False).mul(100).round(2)
        bal = pd.DataFrame({"Count": vc, "Percent %": vp})
        print(bal)

    print("\nData Assessment Completed")

In [9]:
assess_dataframe(new_df, target="unified_target")

Data Assessment Report

Shape of Dataset: 57796 Rows x 33 Columns

Column Information:
                                  DataType  Non-Null Count  Null Count  \
coolant_temp                       float64           40124       17672   
engine_load                        float64           40124       17672   
engine_temp                        float64           40124       17672   
exhaust_temp                       float64           36819       20977   
fuel_consumption                   float64           40124       17672   
fuel_consumption_per_hour          float64           40124       17672   
oil_pressure                       float64           40124       17672   
rpm                                float64           40124       17672   
running_period                     float64           40124       17672   
vibration_level                    float64           40124       17672   
engine_type_0                       object           40124       17672   
engine_type_2-stroke Low-